# 04 — Zero-Shot v3 No Anti-bias

**Amostra:** `data/sample/toldBr_sample_500.csv` (500 tweets estratificados)

Descriçoes das categorias, sem regra de anti-bias.

- Saída: `results/zero_shot_v3_no_antibias.csv`

In [1]:
import time
from pathlib import Path
import polars as pl
import requests
from sklearn.metrics import classification_report, f1_score

OLLAMA_BASE = "http://127.0.0.1:11434"
MODEL = "qwen3.5:9b"
VALID_LABELS = {"NOT_TOXIC", "OBSCENE", "INSULT", "HOMOPHOBIA", "RACISM", "MISOGYNY", "XENOPHOBIA"}
OUTPUT = "../results/zero_shot_v3_no_antibias.csv"

Path("../results").mkdir(exist_ok=True)
sample = pl.read_csv("../data/sample/toldBr_sample_500.csv")
print(f"Tweets: {len(sample)}")

Tweets: 500


In [2]:
def build_prompt(tweet: str) -> str:
    return (
        "Você é um sistema de moderação de conteúdo em português brasileiro.\n\n"
        "Classifique o comentário abaixo em UMA das categorias:\n"
        "- NOT_TOXIC: sem ofensas, críticas construtivas ou conteúdo neutro\n"
        "- OBSCENE: palavrões, linguagem vulgar ou conteúdo sexualmente explícito\n"
        "- INSULT: ataque pessoal ou xingamento direcionado a indivíduo(s)\n"
        "- HOMOPHOBIA: discurso de ódio contra pessoas LGBTQIA+\n"
        "- RACISM: discurso de ódio baseado em raça ou etnia\n"
        "- MISOGYNY: discurso de ódio contra mulheres\n"
        "- XENOPHOBIA: preconceito contra pessoas de outras regiões ou países\n\n"
        "Responda APENAS com o nome da categoria.\n\n"
        f"Comentário: {tweet}\n"
        "Classificação:"
    )

def parse_label(response: str) -> str:
    text = response.strip().upper().replace(" ", "_")
    for label in VALID_LABELS:
        if label in text:
            return label.lower()
    return "unknown"

In [3]:
resultados = []
t0 = time.time()

for i, row in enumerate(sample.iter_rows(named=True)):
    payload = {"model": MODEL, "prompt": build_prompt(row["text"]), "stream": False, "think": False}
    r = requests.post(f"{OLLAMA_BASE}/api/generate", json=payload)
    data = r.json()
    predicao = parse_label(data["response"])
    tps = data["eval_count"] / (data["eval_duration"] / 1e9)
    resultados.append({"text": row["text"], "label": row["label"], "predicao": predicao,
                       "resposta_raw": data["response"].strip(), "tokens_s": round(tps, 1)})
    if (i + 1) % 50 == 0:
        print(f"{i+1}/500 — {time.time()-t0:.0f}s")

df = pl.DataFrame(resultados)
df.write_csv(OUTPUT)
print(f"\nConcluído em {time.time()-t0:.0f}s | UNKNOWN: {(df['predicao']=='unknown').sum()}")

50/500 — 13s


100/500 — 26s


150/500 — 40s


200/500 — 53s


250/500 — 66s


300/500 — 79s


350/500 — 93s


400/500 — 106s


450/500 — 119s


500/500 — 132s

Concluído em 132s | UNKNOWN: 0


In [4]:
y_true, y_pred = df["label"].to_list(), df["predicao"].to_list()
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
print(f"F1-macro: {f1:.4f}\n")
print(classification_report(y_true, y_pred, zero_division=0))

F1-macro: 0.2516

              precision    recall  f1-score   support

  homophobia       0.40      0.50      0.44         4
      insult       0.43      0.08      0.14        36
    misogyny       0.00      0.00      0.00         1
   not_toxic       0.86      0.80      0.83       403
     obscene       0.28      0.47      0.35        55
      racism       0.00      0.00      0.00         0
  xenophobia       0.00      0.00      0.00         1

    accuracy                           0.70       500
   macro avg       0.28      0.26      0.25       500
weighted avg       0.76      0.70      0.72       500

